In [ ]:
!pip install -q langgraph langchain-core langchain-groq langchain-community sentence-transformers faiss-cpu pydantic numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## ⚙️ Prerequisites and Environment Setup

For this lesson, you'll need:

1. **[Groq API key](https://console.groq.com/keys)** (free tier) — for the LLM (Llama 3.1 8b instant).
2. **Local Hugging Face embeddings** — `sentence-transformers/all-MiniLM-L6-v2` runs in-process. No API key, no cloud.
3. Python 3.10+.

> 💡 The first run downloads MiniLM (~90 MB) into your local Hugging Face cache. Subsequent runs are instant.

### Installation


In [ ]:
# Mem0 + Groq + local Hugging Face embeddings (no OpenAI needed)
!pip install -q mem0ai langchain-groq sentence-transformers python-dotenv neo4j

In [ ]:
import os
import json
import numpy as np
from dataclasses import dataclass, field
from datetime import datetime

from dotenv import load_dotenv
load_dotenv()

# Try Colab Secrets first, fall back to manual prompt
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ GROQ_API_KEY loaded from Colab Secrets")
except Exception:
    import getpass
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

from langchain_groq import ChatGroq
from sentence_transformers import SentenceTransformer

print("✅ Imports complete")


✅ GROQ_API_KEY loaded from Colab Secrets
✅ Imports complete


In [ ]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

_probe_dim = len(embedder.encode("hello world"))
print(f"✅ LLM:      Groq llama-3.1-8b-instant")
print(f"✅ Embedder: MiniLM-L6-v2 ({_probe_dim}-dim)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ LLM:      Groq llama-3.1-8b-instant
✅ Embedder: MiniLM-L6-v2 (384-dim)


## 🧠 Phase 1: Working Memory

The simplest form of memory is the **chat history list** — every prior turn is sent back to the LLM so it can reference earlier context. This is *working memory*.

Let's build a minimal agent with chat history only — no long-term store.

In [ ]:
SYSTEM_PROMPT_BASIC = (
    "You are a friendly travel agent. Pay close attention to user details — "
    "destinations, budget, travel dates — and refer back to them when asked."
)


class BasicTravelAgent:
    """Travel agent with working memory only (chat history list)."""

    def __init__(self, system_prompt: str = SYSTEM_PROMPT_BASIC):
        self.system_prompt = system_prompt
        self.history: list[tuple[str, str]] = []  # (role, content)

    def chat(self, user_msg: str) -> str:
        msgs = [("system", self.system_prompt)] + self.history + [("user", user_msg)]
        resp = llm.invoke(msgs).content.strip()
        self.history.append(("user", user_msg))
        self.history.append(("assistant", resp))
        return resp

    def new_session(self):
        self.history.clear()
        print("🔄 Working memory cleared.")


basic = BasicTravelAgent()

# Turn 1 — user shares preferences
print("🤖", basic.chat("I love beach destinations and my budget is $3000."))
print()
# Turn 2 — should recall budget from working memory
print("🤖", basic.chat("What did I say my budget was?"))

🤖 A beach lover, eh? I've got some fantastic options for you. With a budget of $3000, we can explore some amazing beach destinations around the world.

Before I suggest some specific places, can you give me a bit more information? Are you looking for a:

1. Relaxing, laid-back vibe or an action-packed adventure?
2. Tropical island getaway or a more temperate beach destination?
3. Solo trip, couples' retreat, or a family vacation?
4. Specific travel dates in mind (e.g., summer, winter, spring break)?
5. Any preferences for accommodation type (e.g., luxury resort, budget-friendly hotel, vacation rental)?

Let me know, and I'll start crafting a personalized beach vacation just for you!

🤖 You mentioned your budget is $3000.


### 🔍 Test: What happens with a *new* session?

Clearing `history` simulates a brand-new conversation. The agent has **no memory** of the previous turns. This is the core problem long-term memory solves.


In [ ]:
basic.new_session()
print("🤖", basic.chat("What is my budget?"))
print("\n💡 No memory of the previous conversation — it's a fresh session.")

🔄 Working memory cleared.
🤖 I don't have any information about your budget yet. You're welcome to share it with me so I can better assist you in planning your trip.

💡 No memory of the previous conversation — it's a fresh session.


## 🗄️ Phase 2: Long-Term Memory with Vector Embeddings

To remember preferences **across sessions**, we need a persistent store outside the chat history. We'll build a tiny vector memory:

* **add(text)** — encode the text with MiniLM and store the vector + raw text
* **search(query, k)** — encode the query, return the top-k most semantically similar items

Why vectors instead of a plain list? **Semantic recall.** A query about "plant-based meals" will retrieve a stored memory about "vegetarian diet" — even though the words don't overlap.


In [ ]:
@dataclass
class MemoryItem:
    text: str
    created_at: str
    embedding: np.ndarray = field(repr=False)


class VectorMemory:
    """Per-user semantic memory backed by MiniLM embeddings + cosine similarity."""

    def __init__(self):
        self.store: dict[str, list[MemoryItem]] = {}

    def add(self, user_id: str, text: str, dedup_threshold: float = 0.92):
        """Add a memory, skipping if a near-duplicate already exists."""
        vec = embedder.encode(text, normalize_embeddings=True)
        # Dedup — skip if a very similar memory already exists
        for it in self.store.get(user_id, []):
            if float(np.dot(vec, it.embedding)) >= dedup_threshold:
                return  # duplicate
        item = MemoryItem(
            text=text,
            created_at=datetime.now().isoformat(timespec="seconds"),
            embedding=vec,
        )
        self.store.setdefault(user_id, []).append(item)

    def search(self, user_id: str, query: str, k: int = 4) -> list[tuple[float, str]]:
        items = self.store.get(user_id, [])
        if not items:
            return []
        q = embedder.encode(query, normalize_embeddings=True)
        scored = [(float(np.dot(q, it.embedding)), it.text) for it in items]
        scored.sort(key=lambda x: -x[0])
        return scored[:k]

    def reset(self, user_id: str | None = None):
        """Clear memories for one user, or all users if None."""
        if user_id is None:
            self.store.clear()
        else:
            self.store.pop(user_id, None)

    def all(self, user_id: str) -> list[str]:
        return [it.text for it in self.store.get(user_id, [])]


memory = VectorMemory()
print("✅ VectorMemory ready (with dedup + reset)")


✅ VectorMemory ready (with dedup + reset)


### Automatic Preference Extraction

Real systems don't ask the user to manually "save" preferences — they extract them from natural conversation. We use a small Groq prompt to do that: given the user's message, return a JSON list of durable facts worth remembering.

In [ ]:
EXTRACT_PROMPT = (
    "You are a strict fact extractor. From the user message below, extract ONLY "
    "facts the user EXPLICITLY stated about themselves.\n\n"
    "RULES:\n"
    "- Return a JSON array of short factual strings.\n"
    "- Do NOT infer, guess, or generalize. If the user didn't say it, don't include it.\n"
    "- Do NOT include greetings, questions, requests, or meta-comments.\n"
    "- Do NOT repeat facts already implied (e.g. don't add 'traveler' just because they want a hotel).\n"
    "- If the user says nothing factual about themselves, return [].\n\n"
    "INCLUDE categories: budget amounts, dietary restrictions, allergies, "
    "accessibility needs, family composition, loved/hated destinations, travel style.\n\n"
    "Examples:\n"
    "  'Hi, can you recommend a hotel?' → []\n"
    "  'I'm vegetarian and allergic to nuts.' → [\"vegetarian\", \"nut allergy\"]\n"
    "  'Our budget is $700-800 per night.' → [\"budget $700-800/night\"]\n"
    "  'My husband uses a wheelchair.' → [\"husband uses a wheelchair\"]\n"
    "  'Please remember this for next time.' → []\n\n"
    "Message: {msg}\n\nJSON:"
)


def extract_preferences(msg: str) -> list[str]:
    raw = llm.invoke(EXTRACT_PROMPT.format(msg=msg)).content
    try:
        s, e = raw.find("["), raw.rfind("]") + 1
        return [p.strip() for p in json.loads(raw[s:e]) if p.strip()]
    except Exception:
        return []


# Quick sanity check
demo = extract_preferences(
    "I'm vegetarian, allergic to nuts, and my husband uses a wheelchair."
)
print("Extracted:", demo)


Extracted: ['vegetarian', 'nut allergy', 'husband uses a wheelchair']


### Build the Travel Booking Assistant

Now we wrap everything in a `TravelBookingAssistant` that combines:

1. **Working memory** — `chat_history` list per session
2. **Long-term memory** — semantically retrieved from `VectorMemory` on every turn and injected into the system prompt
3. **Auto-extraction** — every user message is passed through `extract_preferences()` and durable facts are stored


In [ ]:
HOTELS = [
    {"name": "Le Meurice Paris",    "location": "Paris, France",    "price": 850, "tags": ["luxury", "romantic", "spa"]},
    {"name": "Four Seasons Maui",   "location": "Maui, Hawaii",     "price": 695, "tags": ["beach", "family", "resort"]},
    {"name": "Aman Tokyo",          "location": "Tokyo, Japan",     "price": 780, "tags": ["luxury", "city", "spa"]},
    {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria",  "price": 420, "tags": ["historic", "accessible", "cultural"]},
    {"name": "Fairmont Whistler",   "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
]

SYSTEM_TEMPLATE = (
    "You are a personalized travel booking assistant.\n\n"
    "User's known preferences (from long-term memory):\n{memories}\n\n"
    "Available hotels (JSON):\n{hotels}\n\n"
    "Rules:\n"
    "- Recommend hotels that match the user's preferences and budget.\n"
    "- NEVER recommend hotels above the user's stated budget.\n"
    "- ONLY recommend hotels from the list above — do NOT invent hotels.\n"
    "- Be concise and friendly."
)


class TravelBookingAssistant:
    def __init__(self, memory: VectorMemory, user_id: str):
        self.memory = memory
        self.user_id = user_id
        self.history: list[tuple[str, str]] = []
        print(f"✅ TravelBookingAssistant ready for user_id='{user_id}'")

    def chat(self, user_msg: str) -> str:
        # 1. Recall relevant long-term memories via semantic search
        hits = self.memory.search(self.user_id, user_msg, k=5)
        mem_block = "\n".join(f"- {t}" for _, t in hits) if hits else "(none yet)"

        # 2. Build the system prompt with memories + hotel catalog
        system = SYSTEM_TEMPLATE.format(
            memories=mem_block,
            hotels=json.dumps(HOTELS, indent=2),
        )

        # 3. Invoke LLM with working memory (history) + new message
        msgs = [("system", system)] + self.history + [("user", user_msg)]
        resp = llm.invoke(msgs).content.strip()

        # 4. Append to working memory
        self.history.append(("user", user_msg))
        self.history.append(("assistant", resp))

        # 5. Extract durable preferences and store in long-term memory
        for pref in extract_preferences(user_msg):
            self.memory.add(self.user_id, pref)

        return resp

    def new_session(self):
        self.history.clear()
        print("🔄 Started a new session (working memory cleared, long-term intact).")

    def get_memories(self) -> list[str]:
        return self.memory.all(self.user_id)


USER_ID = "sarah_johnson_123"

# 🧹 Wipe any stale long-term memory from previous runs of this notebook.
# Comment this out in production — you'd never reset real user memory!
memory.reset(USER_ID)

assistant = TravelBookingAssistant(memory, user_id=USER_ID)
print(f"   Memory for {USER_ID} is empty: {len(assistant.get_memories()) == 0}")


✅ TravelBookingAssistant ready for user_id='sarah_johnson_123'
   Memory for sarah_johnson_123 is empty: True


### Test 1 — First-time user shares preferences

Sarah visits for the first time. Watch the agent absorb her preferences, store them as embeddings, and use them to recommend a hotel.

In [ ]:
r1 = assistant.chat(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night."
)
print("🤖", r1)

🤖 Happy anniversary, Sarah! I'd be delighted to help you find the perfect romantic getaway.

Considering your preferences, I've shortlisted some fantastic options for you:

1. **Le Meurice Paris**: This luxurious hotel in the heart of Paris offers stunning views, a world-class spa, and fine dining experiences. It's also wheelchair accessible, making it an ideal choice for your husband.
2. **Aman Tokyo**: This 5-star hotel in Tokyo boasts breathtaking views of the city, a serene spa, and exceptional dining options. It's also designed with accessibility in mind, ensuring a comfortable stay for your husband.

Both of these hotels fit within your budget of $700-800 per night. Would you like me to provide more details or book one of these options for you?


### Test 2 — Add dietary preferences mid-conversation

Within the same session: working memory holds the Hotel Sacher context; long-term memory absorbs the new dietary facts.

In [ ]:
r2 = assistant.chat(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Please note that for future trips."
)
print("🤖", r2)

🤖 The Hotel Sacher Vienna is a wonderful choice for a romantic getaway.

I've taken note of your dietary preferences and allergy for future trips:

**Known preferences:**

* Romantic destinations
* Fine dining
* Spa experiences
* Accessible accommodations
* Vegetarian diet
* Severe nut allergy

I'll make sure to keep these in mind when suggesting hotels and activities for your future trips.

Now, let's confirm your booking for the Hotel Sacher Vienna. I'll need to check availability and prices for the dates you're planning to visit. Can you please provide me with your travel dates?


## 📊 Memory Analysis — See What Was Learned

Let's peek inside the vector store to see exactly what the extractor decided was worth keeping — and verify that **semantic search** actually works.


In [ ]:
def analyze_extracted_memories():
    print("📈 MEMORY ANALYSIS — What Did the Assistant Learn?")
    print("=" * 60)

    mems = assistant.get_memories()
    print(f"\n🧠 Total preferences stored: {len(mems)}")
    for i, m in enumerate(mems, 1):
        print(f"  {i}. {m}")

    # Test semantic recall — none of these queries share keywords with the stored memories
    print("\n🔎 Semantic recall tests (top hit + similarity score):")
    for q in [
        "plant-based meals",          # should match 'vegetarian'
        "wheelchair access",          # should match 'mobility' / 'accessible'
        "food allergies",             # should match 'nut allergy'
        "price range",                # should match 'budget'
    ]:
        hits = assistant.memory.search(USER_ID, q, k=1)
        if hits:
            score, text = hits[0]
            print(f"  '{q}'  →  ({score:.2f})  {text}")


analyze_extracted_memories()

📈 MEMORY ANALYSIS — What Did the Assistant Learn?

🧠 Total preferences stored: 2
  1. vegetarian
  2. nut allergy

🔎 Semantic recall tests (top hit + similarity score):
  'plant-based meals'  →  (0.43)  vegetarian
  'wheelchair access'  →  (0.04)  vegetarian
  'food allergies'  →  (0.68)  nut allergy
  'price range'  →  (0.10)  vegetarian


### Test 3 — Sarah returns weeks later (new session)

We clear **working memory** to simulate a new conversation. **Long-term memory is intact.** A good agent will pull her preferences via semantic search and personalize the response — even though we never repeat her name, budget, or dietary needs.

In [ ]:
assistant.new_session()  # working memory cleared; long-term memory intact

r3 = assistant.chat(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?"
)
print("🤖", r3)
print(
    "\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
    "even though this is a completely new conversation thread."
)

🔄 Started a new session (working memory cleared, long-term intact).
🤖 I'd be happy to help you find a great hotel. Before I get started, can you tell me a bit more about your preferences? 

Also, I just want to confirm that you're looking for a hotel that can accommodate your vegetarian diet and has no nuts, right?

💡 The agent retrieved Sarah's saved preferences from long-term memory even though this is a completely new conversation thread.


### Test 4 — Continue the new session

Within session 2, working memory now kicks in again — the agent should recall what *it* just recommended.


In [ ]:
r4 = assistant.chat(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?"
)
print("🤖", r4)

🤖 The Four Seasons Maui is a fantastic resort for families. For kids, I'd recommend the following activities:

1. **Snorkeling**: Explore the underwater world at Molokini Crater, a crescent-shaped volcanic crater and marine sanctuary.
2. **Whale Watching**: Take a guided tour to spot humpback whales (seasonal, from December to May).
3. **Luau**: Experience a traditional Hawaiian feast and show, complete with live music and Polynesian dancing.
4. **Beach Games**: Enjoy beach volleyball, paddleboarding, or kayaking in the resort's private cove.
5. **Kids' Club**: The Four Seasons Maui offers a complimentary kids' club, Keiki Kids, with activities like arts and crafts, games, and outdoor adventures.

These activities are sure to create lifelong memories for your little ones!

Would you like me to suggest more activities or help with planning your trip?


## 💬 Phase 3: Interactive Demo — Try It Yourself

Chat freely with the assistant and watch how working + long-term memory cooperate.

Commands:
* `memories` — show everything in the long-term store for this user
* `new` — start a fresh session (clears working memory only)
* `quit` — exit the loop

In [ ]:
def interactive_demo():
    print("💬 Interactive travel agent — chatting as Sarah.")
    print("   Commands: 'memories', 'new', 'quit'\n")
    while True:
        try:
            user_msg = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Demo ended.")
            return

        cmd = user_msg.lower()
        if cmd in ("quit", "exit", "q"):
            print("👋 Thanks for trying the Travel Assistant!")
            return
        elif cmd in ("memories", "memory", "mem"):
            mems = assistant.get_memories()
            print("\n🧠 Long-term memory contents:" if mems else "\n(empty)")
            for i, m in enumerate(mems, 1):
                print(f"  {i}. {m}")
        elif cmd == "new":
            assistant.new_session()
        elif user_msg:
            try:
                print(f"\n🤖 {assistant.chat(user_msg)}")
            except Exception as e:
                print(f"\n⚠️  Error: {e}")


interactive_demo()

💬 Interactive travel agent — chatting as Sarah.
   Commands: 'memories', 'new', 'quit'


You: Hi! Can you recommend a good hotel for our next anniversary?

🤖 Happy anniversary in advance! I'd be delighted to help you find a romantic hotel.

Considering your vegetarian diet and nut allergy, I've filtered the options to recommend a few luxurious hotels that fit your preferences.

Here are my top picks:

1. **Le Meurice Paris**: This 5-star hotel is a masterpiece of French elegance, with stunning views of the Eiffel Tower. Enjoy a romantic dinner at their Michelin-starred restaurant, Le Meurice Alain Ducasse.
2. **Aman Tokyo**: Experience the ultimate in luxury and tranquility at this 5-star hotel, located in the heart of Tokyo. Enjoy a relaxing couples' spa treatment and savor Japanese cuisine at their restaurant.

Both hotels offer exceptional service, beautiful rooms, and a romantic atmosphere. However, please note that Le Meurice Paris is above your stated budget.

Would you like me t

## 🎓 Recap

### What you built
* A **three-layer memory** travel agent using only **Groq + sentence-transformers**
* **Working memory** via a chat-history list
* **Long-term memory** via a tiny `VectorMemory` (MiniLM embeddings + cosine similarity)
* **Automatic preference extraction** — durable facts pulled from natural conversation
* **Semantic recall** — queries match by meaning, not keywords

### Key Takeaways
1. **Working memory = the chat history list.** Same list ⇒ context retained.
2. **New session ⇒ working memory is gone.** Only long-term storage survives.
3. **Embeddings beat exact-match storage** — they let the LLM find relevant facts even when wording differs.
4. **Auto-extraction** keeps users from having to manage memory themselves.

### Production upgrade paths
The in-process numpy store is fine for learning. For real apps swap it for:
* **Qdrant / Chroma / FAISS** — durable, scalable vector search
* **mem0** — purpose-built memory layer with consolidation + conflict resolution
* **Letta / MemGPT** — hierarchical memory (core + recall) with paging
* **Neo4j** — graph memory for entity relationships

### What's next
* Add **memory consolidation** — periodically merge duplicates and resolve contradictions
* Add **decay & forgetting** — Ebbinghaus-style scoring to drop stale items
* Split **episodic vs semantic** memory — timestamped events vs durable facts
* Build **multi-agent systems** that share a memory layer

You're ready to ship agents that get smarter with every conversation. 🚀

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial)

# Persistent Memory for AI Agents with Mem0

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NirDiamant/agents-towards-production/blob/main/tutorials/agent-memory-with-mem0/mem0_tutorial.ipynb)

## 🎯 Introduction

AI agents suffer from a fundamental limitation: **memory amnesia**. They forget everything after each conversation, can't learn from past interactions, and lose valuable context that could make them truly intelligent assistants.

While basic memory solutions store static information, **[Mem0](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=mem0-nir&target=https%3A%2F%2Fmem0.dev%2Fgithub%2Fnir&text=Mem0)** introduces a revolutionary approach: **self-improving memory** that automatically extracts insights, resolves conflicts, and evolves with each interaction. Instead of just storing conversations, Mem0 builds an intelligent knowledge system that learns user preferences and provides contextual understanding.


## 🛠️ What We'll Build

We'll create an intelligent Personal AI Research Assistant that demonstrates Mem0's full capabilities:

It will:

*   Keep track of your preferences for depth, style, and formatting
*   Store research notes, summaries, and insights in vector memory for semantic recall
*   Capture key ideas from your notes and link related concepts inside a graph database
*   Recognize when new questions relate to earlier discussions and surface the right context
*   Use both similarity search and structured relationships to generate more informed answers over time

Instead of restarting from zero every session, the assistant continually builds on what it already knows - helping you form a growing, interconnected understanding of the topics you research.

### Import Required Libraries

In [ ]:
# Core libraries
import os
import getpass

# Mem0 and Groq (instead of OpenAI)
from mem0 import Memory
from langchain_groq import ChatGroq

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Required API Keys

Let's configure your API keys:

In [ ]:
# ── API keys: Groq (LLM) + Qdrant (vector store) ────────────────────────────
# Using Groq (free tier) instead of OpenAI for the LLM.
# Embeddings will use a local Hugging Face model (no API key needed).

import os

# Try Colab Secrets first, fall back to manual prompt
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ GROQ_API_KEY loaded from Colab Secrets")
except Exception:
    import getpass
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

# Optional: Qdrant Cloud (skip → falls back to local in-memory store)
use_qdrant_cloud = input("\nUse Qdrant Cloud? (y/N): ").strip().lower() == "y"
if use_qdrant_cloud:
    import getpass
    if "QDRANT_URL" not in os.environ:
        os.environ["QDRANT_URL"] = input("Enter your Qdrant Cloud URL: ")
    if "QDRANT_API_KEY" not in os.environ:
        os.environ["QDRANT_API_KEY"] = getpass.getpass("Enter your Qdrant API key: ")

print("✅ Setup complete")

✅ GROQ_API_KEY loaded from Colab Secrets

Use Qdrant Cloud? (y/N): N
✅ Setup complete


## Memory Configuration
##  Vector Memory

Now let's configure Mem0's memory system. We're starting with **vector-based memory** - the foundation that makes everything else possible.


### Understanding the Configuration

Each component serves a specific purpose in the memory pipeline:

- **🤖 LLM Provider**: Extracts insights and resolves conflicts
- **📊 Embedder**: Converts text into semantic vectors
- **🗄️ Vector Store**: Stores embeddings for similarity search

In [ ]:
# ── Phase 1 config: Groq LLM + Hugging Face embeddings + Qdrant (optional) ──
# NOTE: If you change embedder dims, RESTART the Colab runtime first
#       (Runtime → Restart session) to clear any cached memory objects.
#
# IMPORTANT: Mem0's memory-extraction prompt is ~9-10K tokens per add().
# `llama-3.1-8b-instant` free tier = 6000 TPM → every add() fails (413 error).
# Use `llama-3.3-70b-versatile` (12000 TPM) for Mem0's internal LLM.

# Default to local Qdrant if the API-keys cell wasn't run (or runtime restarted).
try:
    use_qdrant_cloud  # noqa: F821 — defined in the API-keys cell above
except NameError:
    use_qdrant_cloud = False
    print("ℹ️  use_qdrant_cloud not set — defaulting to LOCAL Qdrant (in-memory).")

config = {
    "llm": {
        "provider": "groq",
        "config": {
            "model": "llama-3.3-70b-versatile",   # bigger TPM budget for extraction
            "temperature": 0.1,
            "max_tokens": 2000,
            "api_key": os.environ["GROQ_API_KEY"],
        },
    },
    "embedder": {
        "provider": "huggingface",
        "config": {
            "model": "sentence-transformers/all-MiniLM-L6-v2",
            "embedding_dims": 384,   # explicit — some Mem0 versions need this
        },
    },
    "version": "v1.1",
}

if use_qdrant_cloud:
    config["vector_store"] = {
        "provider": "qdrant",
        "config": {
            "url": os.environ["QDRANT_URL"],
            "api_key": os.environ["QDRANT_API_KEY"],
            "collection_name": "research_assistant_vectors",
            "embedding_model_dims": 384,  # must match MiniLM
        },
    }
    print("✅ Vector Memory Configuration with Qdrant Cloud")
else:
    # Local Qdrant (in-memory) — must explicitly set 384 dims to match MiniLM
    config["vector_store"] = {
        "provider": "qdrant",
        "config": {
            "collection_name": "research_assistant_local",
            "embedding_model_dims": 384,
            "on_disk": False,  # in-memory
        },
    }
    print("✅ Vector Memory Configuration with LOCAL Qdrant (384-dim)")

# ── Sanity check: verify the embedder actually produces 384-dim vectors ─────
try:
    from sentence_transformers import SentenceTransformer
    _probe = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    _dim = len(_probe.encode("test"))
    print(f"   Embedder probe: MiniLM produces {_dim}-dim vectors ✅" if _dim == 384
          else f"    Unexpected dim: {_dim}")
except Exception as e:
    print(f"   Could not probe embedder: {e}")

print(f"   LLM (Mem0 extraction): Groq llama-3.3-70b-versatile (12K TPM)")
print(f"   Embedder:              HuggingFace MiniLM-L6-v2 (384-dim, local)")


✅ Vector Memory Configuration with LOCAL Qdrant (384-dim)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   Embedder probe: MiniLM produces 384-dim vectors ✅
   LLM (Mem0 extraction): Groq llama-3.3-70b-versatile (12K TPM)
   Embedder:              HuggingFace MiniLM-L6-v2 (384-dim, local)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Initialize Memory System

Now let's create our memory instance - the brain of our research assistant:

In [ ]:
# Clear any previously cached memory object to avoid dim mismatches
try:
    del memory
except NameError:
    pass

try:
    memory = Memory.from_config(config)
    print("✅ Memory system initialized successfully!")

    # Sanity check: write + read to verify embedder dim matches store
    memory.add("test entry to verify embedder dimensions", user_id="_probe_")
    _r = memory.search(query="test", filters={"user_id": "_probe_"})
    print(f"   Probe search returned: {len(_r.get('results', _r)) if isinstance(_r, dict) else len(_r)} item(s)")
except Exception as e:
    print(f"❌ Error initializing memory: {e}")
    print("   → Try: Runtime → Restart session, then re-run all cells from the top")


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Memory system initialized successfully!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/mem0/vector_stores/qdrant.py:169: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self.client.create_payload_index(


   Probe search returned: 1 item(s)


### Core Mem0 Operations

Before we build the assistant, let's understand Mem0's main operations:

`memory.add()` - Stores new information
- Automatically extracts key facts from conversations
- Handles deduplication and conflicts
- Stores memories with user-specific isolation

![mem0_add_architecture](Assets/mem0_add_architecture.jpg)

`memory.search()` - Retrieves relevant memories  
- Finds semantically similar content (not just keyword matching)
- Returns ranked results by relevance
- Supports filtering by user_id

![mem0_search_architecture](Assets/mem0_search_architecture.jpg)

**Other operations** (not used in this tutorial):
- `memory.update()` - Modify existing memories
- `memory.delete()` - Remove specific memories

In [ ]:
import time

class PersonalResearchAssistant:

    def __init__(self, memory_instance):
        # Chat model = small/fast 8b (the assistant's replies).
        # Mem0 internally uses 70b for memory extraction (see config cell).
        self.llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.1)
        self.memory = memory_instance
        print("Research Assistant initialized with Mem0 memory + Groq LLM!")

    def ask(self, question, user_id):

        # Search for relevant memories
        previous_memories = self.search_memories(question, user_id=user_id)

        system_message = (
            "You are a personal AI Research Assistant. Help users with research "
            "questions, remember their interests, and provide contextual recommendations."
        )

        if previous_memories:
            memory_context = ", ".join(previous_memories)
            prompt = f"{system_message}\n\nUser input: {question}\nPrevious memories: {memory_context}"
        else:
            prompt = f"{system_message}\n\nUser input: {question}"

        try:
            response = self.llm.invoke(prompt)
            answer = response.content if hasattr(response, "content") else str(response)

            # Store the question — retry once on rate limit
            self._safe_add(question, user_id)
            return answer

        except Exception as e:
            return f"Encountered an error: {e}"

    def _safe_add(self, question, user_id, retries=2):
        for attempt in range(retries + 1):
            try:
                self.memory.add(question, user_id=user_id, metadata={"category": "research"})
                return
            except Exception as e:
                msg = str(e).lower()
                if "rate_limit" in msg or "413" in msg or "tokens per minute" in msg:
                    if attempt < retries:
                        wait = 30 * (attempt + 1)
                        print(f"   ⏳ Rate-limited by Groq. Waiting {wait}s before retry...")
                        time.sleep(wait)
                        continue
                print(f"   ⚠️  memory.add() failed: {e}")
                return

    def get_memories(self, user_id):
        try:
            try:
                memories = self.memory.get_all(filters={"user_id": user_id})
            except TypeError:
                memories = self.memory.get_all(user_id=user_id)
            if isinstance(memories, dict) and 'results' in memories:
                return [m['memory'] for m in memories['results']]
            elif isinstance(memories, list):
                return [m['memory'] for m in memories]
            return []
        except Exception as e:
            print(f"Error retrieving memories: {e}")
            return []

    def search_memories(self, query, user_id):
        try:
            try:
                results = self.memory.search(query=query, filters={"user_id": user_id})
            except TypeError:
                results = self.memory.search(query=query, user_id=user_id)
            if isinstance(results, dict) and 'results' in results:
                return [r['memory'] for r in results['results']]
            elif isinstance(results, list):
                return [r['memory'] for r in results]
            return []
        except Exception as e:
            print(f"Error searching memories: {e}")
            return []


# ── Instantiate the assistant using the memory created earlier ──────────────
assistant = PersonalResearchAssistant(memory)


Research Assistant initialized with Mem0 memory + Groq LLM!


### Test 1: First Interaction - Knowledge Extraction

Let's start with a basic research interest:

In [ ]:
response1 = assistant.ask(
    "I'm interested in transformer architectures for natural language processing. "
    "Can you help me find recent papers on this topic?",
    user_id="researcher"
)

print(f"Assistant: {response1}")

Assistant: I'd be happy to help you with recent papers on transformer architectures for natural language processing.

**Recent Papers (2020-2023)**

Here are some recent papers on transformer architectures for NLP:

1. **"Longformer: The Extreme Long Document Transformers"** by Iz Beltagy, Matthew E. Peters, and Arman Cohan (2020) - This paper introduces the Longformer, a transformer architecture designed to handle long documents.
2. **"Reformer: The Efficient Transformer"** by Nikita Kitaev, Łukasz Kaiser, and Anselm Levskaya (2020) - This paper proposes the Reformer, a transformer architecture that uses reversible transformations to reduce memory usage.
3. **"Big Bird: Transformers for Longer Documents"** by Yinhan Liu, Pengcheng He, Weizhen Qi, et al. (2021) - This paper introduces the Big Bird, a transformer architecture designed to handle long documents.
4. **"Sparse Transformers"** by Raffel, Colin, et al. (2020) - This paper proposes the sparse transformer, a transformer archite

### Test 2: Building Context - Watch Memory Connect Ideas

In [ ]:
response2 = assistant.ask(
    "I prefer papers that include practical implementation details and code examples. "
    "Theoretical papers without code are less useful for my work.",
    user_id="researcher"
)

print(f"Assistant: {response2}")

Assistant: Based on your preference for papers with practical implementation details and code examples, and your interest in transformer architectures for natural language processing, I've searched for recent papers that fit your criteria. Here are some recommendations:

1. **"Longformer: The Extreme Long Document Transformers"** by Iz Beltagy, Kyle Lo, and Arman Cohan (2020)

This paper introduces the Longformer, a transformer architecture designed to handle long documents. The authors provide a detailed implementation in PyTorch and discuss the challenges of processing long documents.

2. **"Big Bird: Transformers for Longer Documents"** by Yinhan Liu, Pengcheng He, Weizhen Qi, et al. (2020)

Big Bird is another transformer architecture designed for long documents. The authors provide a PyTorch implementation and discuss the trade-offs between model size and performance.

3. **"Reformer: The Efficient Transformer"** by Nikita Kitaev, Łukasz Kaiser, and Anselm Levskaya (2020)

Reforme

### Test 3: Semantic Search - Intelligence Beyond Keywords

In [ ]:
response3 = assistant.ask(
    "What about BERT and GPT models? Are they related to my research interests?",
    user_id="researcher"
)

print(f"Assistant: {response3}")

Assistant: Based on our previous conversation, I recall that you're interested in transformer architectures for natural language processing. BERT and GPT are indeed related to this topic.

**BERT (Bidirectional Encoder Representations from Transformers)**

BERT is a transformer-based language model developed by Google in 2018. It's a pre-trained language model that uses a multi-layer bidirectional transformer encoder to generate contextualized representations of words in a sentence. BERT has achieved state-of-the-art results in various natural language processing tasks, such as question answering, sentiment analysis, and text classification.

**GPT (Generative Pre-trained Transformer)**

GPT is a transformer-based language model developed by OpenAI in 2018. It's a pre-trained language model that uses a multi-layer transformer decoder to generate coherent and context-specific text. GPT has achieved impressive results in natural language generation tasks, such as text summarization, chat

### Test 4: Conflict Resolution - Intelligent Memory Management

In [ ]:
response4 = assistant.ask(
    "Actually, I also need to understand the theoretical foundations of attention mechanisms. "
    "Can you recommend some foundational theory papers?",
    user_id="researcher"
)

print(f"Assistant: {response4}")

Assistant: Based on your previous interests and the new request for theoretical foundations of attention mechanisms, I've curated a list of foundational theory papers and recent research papers on transformer architectures for natural language processing. I've also included some practical implementation details and code examples for BERT and GPT models.

**Foundational Theory Papers:**

1. **"A Mathematical Theory of Communication" by Claude Shannon (1948)**: This classic paper lays the foundation for information theory, which is essential for understanding attention mechanisms.
2. **"Attention and Affect in the Perception of Faces" by James E. Cutting (1978)**: This paper introduces the concept of attention in the context of face perception, which is relevant to the attention mechanisms used in transformer architectures.
3. **"The Attention Revolution: Toward a New Realization of Mind" by Daniel J. Siegel (2010)**: This book provides an in-depth exploration of attention and its role i

## 📊 Memory Analysis - See What Was Learned

Let's examine what our assistant learned from these interactions:

In [ ]:
def analyze_extracted_memories():

    print("📈 MEMORY ANALYSIS - What Did the Assistant Learn?")
    print("=" * 60)

    # Get all memories
    all_memories = assistant.get_memories(user_id="researcher")

    if all_memories:
        print(f"\n🧠 Total memories extracted: {len(all_memories)}")
        print(f"\n📚 Key insights Mem0 learned about you:")

        for i, memory in enumerate(all_memories, 1):
            print(f"\n{i}. {memory}")

        # Test semantic search
        print(f"\n🔍 Testing Semantic Search Capabilities:")

        test_queries = [
            "neural networks",           # Should connect to transformers
            "code implementations",      # Should find practical preferences
            "attention mechanisms",      # Should connect to transformer interest
            "deep learning papers"       # Should find research interests
        ]

        for query in test_queries:
            related_memories = assistant.search_memories(query, user_id="researcher")
            print(f"\n   🔎 Query: '{query}'")
            print(f"      Found {len(related_memories)} related memories")
            if related_memories:
                print(f"      Top match: {related_memories[0][:100]}...")

    else:
        print("\n⚠️ No memories found. Try running the interaction tests first.")

    return all_memories

# Run the analysis
memories = analyze_extracted_memories()

📈 MEMORY ANALYSIS - What Did the Assistant Learn?

🧠 Total memories extracted: 4

📚 Key insights Mem0 learned about you:

1. User is interested in transformer architectures for natural language processing and is looking for recent papers on this topic

2. User prefers transformer architecture papers with practical implementation details and code examples for natural language processing

3. User needs to understand the theoretical foundations of attention mechanisms in transformer architectures for natural language processing

4. User is interested in BERT and GPT models in relation to their research on transformer architectures for natural language processing

🔍 Testing Semantic Search Capabilities:

   🔎 Query: 'neural networks'
      Found 4 related memories
      Top match: User needs to understand the theoretical foundations of attention mechanisms in transformer architec...

   🔎 Query: 'code implementations'
      Found 4 related memories
      Top match: User prefers transformer

In [ ]:
def interactive_demo(user_id="researcher"):
    print(f"💬 Interactive demo — chatting as user_id='{user_id}'")
    print("   Type 'memories' to see what's been learned, or 'quit' to exit.\n")
    while True:
        try:
            question = input("\nYou: ")

            if question.lower() in ['quit', 'exit', 'q']:
                print("\n Thanks for trying the Research Assistant!")
                break
            elif question.lower() == 'memories':
                memories = assistant.get_memories(user_id=user_id)
                if memories:
                    print(f"\n Here's what I've learned about your research interests:")
                    for i, memory in enumerate(memories, 1):
                        print(f"   {i}. {memory}")
                else:
                    print("\n No memories yet. Start asking about your research interests!")
            elif question.strip():
                response = assistant.ask(question, user_id=user_id)
                print(f"\n Assistant: {response}")

        except KeyboardInterrupt:
            print("\n\n Demo ended. Thanks for trying the Research Assistant!")
            break
        except Exception as e:
            print(f"\n Error: {e}")

# Run the interactive demo (uses the same user_id as the earlier tests)
interactive_demo(user_id="researcher")


💬 Interactive demo — chatting as user_id='researcher'
   Type 'memories' to see what's been learned, or 'quit' to exit.


You: What papers should I read next about LLMs?

 Assistant: Based on your previous interests and preferences, I've curated a list of recent papers on transformer architectures for natural language processing (NLP) that you might find interesting. Here are some recommendations:

1. **"Longformer: The Extreme Long Document Transformers"** by Iz Beltagy, Matthew E. Peters, and Arman Cohan (2020)

This paper introduces the Longformer, a transformer architecture designed to handle long documents. It provides a detailed explanation of the architecture and includes code examples in PyTorch.

2. **"Big Bird: Transformers for Longer Documents"** by Yinhan Liu, Pengcheng He, Weizhen Qi, Chengxu Zhu, Jianfeng Gao, Li Dong, Xia Song, Jian Yin, and Eric P. Xing (2020)

Big Bird is another transformer architecture designed for long documents. The paper provides a thorough explan

ERROR:mem0.memory.main:LLM extraction failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01km3pv4wzejr8pcb9ev6gb1mr` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 93605, Requested 8237. Please try again in 26m31.487999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}



 Assistant: It was nice assisting you. If you ever need help with anything or want to explore a new topic, feel free to come back and ask. I'll be here to assist you.

Before we part ways, I just wanted to check if there's anything I can help you remember or follow up on from our previous conversations. We didn't have any prior conversations, but I can start fresh if you'd like.

If you're interested in exploring new topics or want recommendations based on your interests, I can suggest some areas to explore. Just let me know what you're in the mood for (e.g., science, history, entertainment, etc.).


## 🚀 Phase 2: Enhancing with Graph Memory Capabilities

Now that you've mastered **vector-based memory** and seen its power, let's enhance our system with **graph capabilities**. This addition will enable explicit relationship mapping between entities, concepts, and research domains.

### Why Add Graph Memory?

**Vector memory excels at**:

- Finding semantically similar content
- "Show me papers like this one"
- Understanding conceptual similarity

**Graph memory adds**:

- Explicit entity relationships
- "Who collaborated with whom?"
- "How did Paper A influence Paper B?"
- Research lineage and citation networks